# Gomoku Zero

Train a 15×15 Gomoku AI with Renju forbidden-hand rules using MCTS + ResNet.

## 1. Setup: Mount Drive & Install Dependencies

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_DIR = "/content/gomoku-zero"
CHECKPOINT_DIR = "/content/drive/MyDrive/Colab/gomoku-zero/checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# Clone or pull repo
if not os.path.exists(PROJECT_DIR):
    !git clone https://github.com/hikaru-im/gomoku-zero.git {PROJECT_DIR}
else:
    %cd {PROJECT_DIR} && git pull

%pip install torch torchvision numpy cython

## 2. Compile C++ Core (Cython)

**Must re-run after every `git pull`** that touches `bitboard.cpp` or `cboard.pyx`.

In [ ]:
%cd {PROJECT_DIR}
!rm -rf build
!python core_logic/setup.py build_ext --inplace

# Verify compilation
from core_logic.cboard import PyBoard
board = PyBoard()
print(f"Board initialized: {board.num_stones} stones, current player: {board.current_player}")
print(f"Legal moves (first 10): {board.legal_moves()[:10]}")
print("C++ core compiled successfully!")

## 3. Smoke Test (optional but recommended)

Verify the full pipeline works before starting a long training run.

In [ ]:
import torch
import numpy as np
import sys
sys.path.insert(0, PROJECT_DIR)

from core_logic.cboard import PyBoard
from neural_net.resnet import ResNetGomoku
from search.mcts_batch import MCTSBatch, Node

device = "cuda" if torch.cuda.is_available() else "cpu"

print("=== Test 1: Board ===")
board = PyBoard()
board.play_move(7, 7, 1)
board.play_move(8, 8, 2)
print(f"Stones: {board.num_stones}, current: {board.current_player}")
print(f"Black at (7,7): {board.is_black(7,7)}, White at (8,8): {board.is_white(8,8)}")

print("\n=== Test 2: Win Detection ===")
board2 = PyBoard()
for i in range(5):
    board2.play_move(i, 0, 1)
    if i < 4:
        board2.play_move(i, 1, 2)
print(f"Black wins: {board2.check_win(1)}")

print("\n=== Test 3: Forbidden Move ===")
board3 = PyBoard()
print(f"Center forbidden for black: {board3.is_forbidden(7, 7)}")

print("\n=== Test 4: Network ===")
net = ResNetGomoku(num_blocks=4, num_filters=32).to(device)
state = np.zeros((1, 3, 15, 15), dtype=np.float32)
logits, value = net.predict(torch.from_numpy(state).to(device))
print(f"Policy shape: {logits.shape}, Value shape: {value.shape}")
print(f"Value: {value.item():.4f}")

print("\n=== Test 5: MCTS ===")
board4 = PyBoard()
mcts = MCTSBatch(num_simulations=64)
root = Node()
probs = mcts.search(root, board4, net, device)
print(f"Policy sum: {probs.sum():.4f}, non-zero: {(probs > 0).sum()}")
print(f"Best move: ({np.argmax(probs) % 15}, {np.argmax(probs) // 15})")

print("\n=== Test 6: Illegal move rejection ===")
board5 = PyBoard()
board5.play_move(7, 7, 1)
result = board5.play_move(7, 7, 2)  # occupied
print(f"Play on occupied cell rejected: {result == False}")
result2 = board5.play_move(0, 0, 1)  # wrong player
print(f"Play with wrong player rejected: {result2 == False}")

print("\nAll smoke tests passed!")

## 4. Start Training

- `trainer.resume()` loads the latest checkpoint (model + optimizer + scheduler + iteration + replay buffer).
- First run: no checkpoint exists, training starts from scratch.
- After a disconnect: re-run cells 1-2 (Setup + Compile), then run this cell to resume.

In [ ]:
import torch
import sys
sys.path.insert(0, PROJECT_DIR)

from training.trainer import GomokuTrainer

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

trainer = GomokuTrainer(
    checkpoint_dir=CHECKPOINT_DIR,
    num_blocks=10,
    num_filters=128,
    lr=2e-3,
    batch_size=512,
    self_play_games=50,
    training_steps=500,
    mcts_simulations=200,
    eval_games=100,
    temperature_threshold=30,
    device=device,
)

trainer.resume()
trainer.run(max_iterations=1000)
print("Training complete!")

## 5. Notes

- **Checkpoint files**: `best_model.pt` (best evaluated), `latest.pt` (crash recovery), `latest_buffer.pt` (replay buffer), `model_iter_N.pt` (per-iteration snapshots).
- **Resume from scratch**: delete all files in `CHECKPOINT_DIR` and re-run.
- **Colab disconnects**: just re-run cells 1, 2, and 4. `resume()` will pick up where you left off.